In [2]:
import os
import pandas as pd
import hashlib

train_dir = "../dataset/processed/train"
test_dir = "../dataset/processed/test"


In [3]:
data = []

for emotion in sorted(os.listdir(train_dir)):
    emotion_path = os.path.join(train_dir, emotion)

    for image_name in os.listdir(emotion_path):
        image_path = os.path.join(emotion_path, image_name)

        data.append({
            "image_path": image_path,
            "label": emotion
        })

train_df = pd.DataFrame(data)

print("Jumlah data:", len(train_df))
train_df.head()

Jumlah data: 27434


,image_path,label
0,../dataset/processed/train\angry\Training_1011...,angry
1,../dataset/processed/train\angry\Training_1012...,angry
2,../dataset/processed/train\angry\Training_1016...,angry
3,../dataset/processed/train\angry\Training_1021...,angry
4,../dataset/processed/train\angry\Training_1026...,angry


In [4]:
train_df["hash"] = train_df["image_path"].apply(
    lambda x: hashlib.md5(open(x, "rb").read()).hexdigest()
)

print("Jumlah data       :", len(train_df))
print("Jumlah hash unik   :", train_df["hash"].nunique())
print("Jumlah duplicate   :", len(train_df) - train_df["hash"].nunique())

Jumlah data       : 27434
Jumlah hash unik   : 27434
Jumlah duplicate   : 0


In [5]:
print(train_df["label"].value_counts().sort_index())

label
angry       3833
disgust      379
fear        3885
happy       7085
neutral     4867
sad         4716
surprise    2669
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

train_split, val_split = train_test_split(
    train_df,
    test_size=0.20,
    random_state=42,
    stratify=train_df["label"]
)

train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)

print("Jumlah data training   :", len(train_split))
print("Jumlah data validation :", len(val_split))

Jumlah data training   : 21947
Jumlah data validation : 5487


In [7]:
print("Distribusi TRAIN:")
print(train_split["label"].value_counts(normalize=True).sort_index())

print("\nDistribusi VALIDATION:")
print(val_split["label"].value_counts(normalize=True).sort_index())

Distribusi TRAIN:
label
angry       0.139700
disgust     0.013806
fear        0.141614
happy       0.258259
neutral     0.177427
sad         0.171914
surprise    0.097280
Name: proportion, dtype: float64

Distribusi VALIDATION:
label
angry       0.139785
disgust     0.013851
fear        0.141607
happy       0.258247
neutral     0.177328
sad         0.171861
surprise    0.097321
Name: proportion, dtype: float64


In [15]:
train_split.to_csv(
    "../dataset/train_split.csv",
    index=False
)

val_split.to_csv(
    "../dataset/val_split.csv",
    index=False
)

print("Train split dan validation split berhasil disimpan.")

Train split dan validation split berhasil disimpan.


In [8]:
class_names = sorted(train_df["label"].unique())

print("Class names:")
for i, class_name in enumerate(class_names):
    print(i, "→", class_name)

Class names:
0 → angry
1 → disgust
2 → fear
3 → happy
4 → neutral
5 → sad
6 → surprise


In [9]:
label_to_index = {
    class_name: index
    for index, class_name in enumerate(class_names)
}

train_split["label_index"] = train_split["label"].map(label_to_index)
val_split["label_index"] = val_split["label"].map(label_to_index)

print(train_split.head())

                                          image_path    label  \
0  ../dataset/processed/train\neutral\Training_19...  neutral   
1  ../dataset/processed/train\happy\Training_5977...    happy   
2  ../dataset/processed/train\angry\Training_2226...    angry   
3  ../dataset/processed/train\angry\Training_5622...    angry   
4  ../dataset/processed/train\angry\Training_7324...    angry   

                               hash  label_index  
0  59e9cf176ae7903974f8e2445ef185c9            4  
1  2222a4629a48c362a95cbd5f5ad7fa74            3  
2  b3f7d04564912e590c91a62d5a10c179            0  
3  7331dc3297e9693ded4c6a56a1dd626d            0  
4  9421f59d2c9679c60e5289cc52e8f468            0  


In [10]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [11]:
IMG_SIZE = (48, 48)


def load_and_preprocess_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=1)

    image = tf.cast(image, tf.float32)
    image = image / 255.0

    image = tf.reshape(image, [48, 48, 1])

    return image, label

In [12]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        train_split["image_path"].values,
        train_split["label_index"].values
    )
)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (
        val_split["image_path"].values,
        val_split["label_index"].values
    )
)

train_dataset = train_dataset.map(
    load_and_preprocess_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_dataset = val_dataset.map(
    load_and_preprocess_image,
    num_parallel_calls=tf.data.AUTOTUNE
)


In [13]:
BATCH_SIZE = 32

train_dataset = train_dataset.shuffle(
    buffer_size=len(train_split),
    seed=42
)

train_dataset = train_dataset.batch(BATCH_SIZE)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

val_dataset = val_dataset.batch(BATCH_SIZE)
val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)

In [14]:
images, labels = next(iter(train_dataset))

print("Shape gambar :", images.shape)
print("Shape label  :", labels.shape)
print("Nilai minimum:", tf.reduce_min(images).numpy())
print("Nilai maksimum:", tf.reduce_max(images).numpy())

Shape gambar : (32, 48, 48, 1)
Shape label  : (32,)
Nilai minimum: 0.0
Nilai maksimum: 1.0
